# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show main metadata fields
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Date Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Keywords: {dataset.metadata.keywords}")


## 2. Data Overview
Review available record sets and their fields. Each record set and field is referenced by its `@id`.


In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.metadata.recordSets or []

if not record_sets:
    print("This dataset does not define record sets in the top-level Croissant metadata. Attempting to infer possible data downloads or file objects.")
    # Try to infer from distribution objects
    distributions = dataset.metadata.distribution if hasattr(dataset.metadata, 'distribution') else []
    record_sets_from_distributions = []
    for dist in distributions:
        # If the distribution object fits a CSV or data structure, add it
        if hasattr(dist, '@id'):
            print(f"Possible record set from distribution: {dist['@id']}")
            record_sets_from_distributions.append(dist['@id'])
    # If no explicit record sets, use the first distribution as our record set
    if record_sets_from_distributions:
        record_sets = record_sets_from_distributions
    else:
        print("No explicit data files or record sets found. Please inspect the dataset schema or specification.")

# If the record_sets are object references, print their IDs
for rs in record_sets:
    print(f"RecordSet @id: {rs}")

## 3. Data Extraction
Load data from a specific record set (referenced by its `@id`) into a DataFrame for analysis.

In [ ]:
# Define the record set to use. From previous cell, @id of distribution is likely the record set @id.

selected_record_set = record_sets[0]  # Use the first found record set/distribution @id
print(f"Selected Record Set @id: {selected_record_set}")

# Load records for this record set
records = list(dataset.records(record_set=selected_record_set))

df = pd.DataFrame(records)
print(f"Loaded DataFrame columns from Record Set {selected_record_set}:\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Find likely numeric fields for EDA
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or (df[col].dtype == 'object' and df[col].str.replace('.', '', 1).str.isnumeric().any())]
print(f"Possible numeric fields: {numeric_candidates}")

# Let's pick a field; if there's 'MW [kDa]' or 'SequenceCoverage [%]', prefer that. Fallback to any numeric field.
preferred_numeric_fields = ['MW [kDa]', 'SequenceCoverage [%]', 'Peptides (total)', 'iBAQ_Sample_1', 'Abundance']
numeric_field = next((f for f in preferred_numeric_fields if f in df.columns), (numeric_candidates[0] if numeric_candidates else None))

if numeric_field is None:
    print("No numeric field found in dataset.")
else:
    print(f"Using numeric field: {numeric_field}")
    # Try to convert to float just in case
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top quartile): {len(filtered_df)} rows")
    
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First few normalized values for {numeric_field}:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a categorical field, e.g., 'Description' or first non-numeric field
    group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
    group_field = next((c for c in group_candidates if 'Description' in c or 'Type' in c), (group_candidates[0] if group_candidates else None))
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field} (first 5 groups):")
        print(grouped_df.head())
    else:
        print("No categorical group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field:
        plt.figure(figsize=(10,5))
        # Show top 10 groups by frequency
        top_groups = df[group_field].value_counts().head(10).index
        sns.boxplot(x=df[group_field][df[group_field].isin(top_groups)], y=df[numeric_field][df[group_field].isin(top_groups)])
        plt.title(f"{numeric_field} by {group_field} (Top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> dataset, explored its structure using the `mlcroissant` library, and performed essential exploratory analyses and visualizations. The dataset details protein abundance, modifications, and sequences extracted from human samples using mass spectrometry.

We identified key numeric fields such as molecular weight (`MW [kDa]`) or sequence coverage, applied filtering and normalization, grouped and summarized by descriptive or type fields, and visualized critical data relationships. This provides a foundation for further bioinformatic or proteomics analyses using the FAIR<sup>2</sup> dataset schema.
